# Scene-graph demo — place → move → place → hover → connect

This notebook walks through the framework end-to-end **without an LLM**, driving the same operands the Executor would pick. The point is to show:

1. The **scene graph** (deterministic, JSON-backed) updating after every operand call.
2. **Inter-operation scanning** — the bbox of the selected shape gets re-detected from CV after geometry changes, and reconciled back into the matching scene-graph object.
3. The new shape-manipulation operands: `move_shape`, `hover_object`, `connect_shapes`.

## Prerequisites

- draw.io running in a window where pyautogui can drive it (focus it during the countdown).
- On macOS, *Screen Recording* permission for the terminal running Jupyter.
- The current `state/ui_graph.json` calibrated for your drawio window.

In [19]:
# --- 0. Setup --------------------------------------------------------
import os, sys, time, json
sys.path.insert(0, os.path.abspath('..'))

from core import config
from core.tools import dispatch, TOOL_CATALOG
from core.state import scene_graph as sg

print(f'Loaded {len(TOOL_CATALOG)} operands.')
for name in ['place_shape','type_label','press_escape','move_shape','hover_object','connect_shapes']:
    assert name in TOOL_CATALOG, f'missing {name}'
    print(f'  ✓ {name}')

Loaded 25 operands.
  ✓ place_shape
  ✓ type_label
  ✓ press_escape
  ✓ move_shape
  ✓ hover_object
  ✓ connect_shapes


In [20]:
# --- 1. Reset scene graph + share ui_graph across the notebook -------
sg.reset()

# One ui_graph dict, threaded through every dispatch() call so the
# scene_graph and selected_handles persist across cells.
g = config.ui_graph()
g['scene_graph'] = sg.load()

def show_scene():
    print(sg.summary_for_prompt(g['scene_graph']))

show_scene()

_Canvas is empty._


In [21]:
# --- 2. Focus drawio + clean canvas ---------------------------------
# Switch to your draw.io window now. After the countdown the cells
# below will start driving pyautogui clicks against whatever has focus.
# The clean phase deselects + undoes ~40 ops to clear any leftover
# shapes from prior runs so each cell starts from a known state.
#
# IMPORTANT: re-run this cell whenever you want to restart the demo —
# without it the click on the sidebar can land while a prior shape is
# selected, and drawio interprets the click against that context
# (which has caused 'wrong shape gets placed' bugs in the past).
import pyautogui
print('Switch to draw.io NOW.')
for i in range(5, 0, -1):
    print(f'  {i}s …')
    time.sleep(1)
print('GO — cleaning canvas …')
pyautogui.click(1100, 800); time.sleep(0.3)   # click empty canvas to deselect
for _ in range(40):
    pyautogui.hotkey('command', 'z')
    time.sleep(0.04)
pyautogui.click(1100, 800); time.sleep(0.3)
print('clean.')


Switch to draw.io NOW.
  5s …
  4s …
  3s …
  2s …
  1s …
GO — cleaning canvas …
clean.


In [22]:
# --- 3. Place rectangle A and label it 'Source' ----------------------
r = dispatch('place_shape', {'tool_name': 'Rectangle_Tool'}, ui_graph=g)
print('place_shape:', r)
time.sleep(0.3)
r = dispatch('type_label', {'text': 'Source'}, ui_graph=g)
print('type_label :', r)
r = dispatch('press_escape', {}, ui_graph=g)
print('press_escape:', r)

print('\n--- scene_graph after place A ---')
show_scene()

  [L0] place_shape('Rectangle_Tool') → click (36, 322) + Enter
place_shape: {'status': 'ok', 'tool': 'place_shape', 'tool_name': 'Rectangle_Tool', 'x': 36, 'y': 322, 'scene_object_id': 'obj_001', 'level': 0}
  [L0] type_label('Source')
type_label : {'status': 'ok', 'tool': 'type_label', 'text': 'Source', 'level': 0}
  [L0] press_escape
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
press_escape: {'status': 'ok', 'tool': 'press_escape', 'level': 0}

--- scene_graph after place A ---
**Objects (1):**
  - `obj_001` Rectangle "Source"  bbox=[708,515,84x42]  *SELECTED*

_(scene_graph op #3, last op: press_escape)_


In [23]:
# --- 4. Move A upward by 120 logical px ------------------------------
r = dispatch('move_shape', {'direction': 'n', 'amount': 120}, ui_graph=g)
print('move_shape:', r)
print('\n--- scene_graph after moving A north ---')
show_scene()

  [L0] move_shape('n', 120) → drag (750,536) → (750,416)
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
move_shape: {'status': 'ok', 'tool': 'move_shape', 'direction': 'n', 'amount': 120, 'from': [750, 536], 'to': [750, 416], 'level': 0}

--- scene_graph after moving A north ---
**Objects (1):**
  - `obj_001` Rectangle "Source"  bbox=[704,396,84x42]  *SELECTED*

_(scene_graph op #4, last op: move_shape:n)_


In [24]:
# --- 5. Place rectangle B and label it 'Target' ----------------------
# B becomes the new selection; A is still in the scene graph at its
# moved position, and the inter-op scanner will fill in B's bbox after
# escape.
r = dispatch('place_shape', {'tool_name': 'Rectangle_Tool'}, ui_graph=g)
print('place_shape:', r)
time.sleep(0.3)
r = dispatch('type_label', {'text': 'Target'}, ui_graph=g)
print('type_label :', r)
r = dispatch('press_escape', {}, ui_graph=g)
print('press_escape:', r)

print('\n--- scene_graph after placing B ---')
show_scene()

  [L0] place_shape('Rectangle_Tool') → click (36, 322) + Enter
place_shape: {'status': 'ok', 'tool': 'place_shape', 'tool_name': 'Rectangle_Tool', 'x': 36, 'y': 322, 'scene_object_id': 'obj_002', 'level': 0}
  [L0] type_label('Target')
type_label : {'status': 'ok', 'tool': 'type_label', 'text': 'Target', 'level': 0}
  [L0] press_escape
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
press_escape: {'status': 'ok', 'tool': 'press_escape', 'level': 0}

--- scene_graph after placing B ---
**Objects (2):**
  - `obj_001` Rectangle "Source"  bbox=[704,396,84x42]
  - `obj_002` Rectangle "Target"  bbox=[708,515,84x42]  *SELECTED*

_(scene_graph op #7, last op: press_escape)_


In [25]:
# --- 6. Hover B without clicking ------------------------------------
# This is the 'hover without selecting' step the demo description asks
# for. drawio responds to hover by drawing the 4 directional extend
# arrows around the shape. The mouse stops on the shape; no click.

scene = g['scene_graph']
objs = scene['objects']
obj_a = next(o for o in objs if o['label'] == 'Source')
obj_b = next(o for o in objs if o['label'] == 'Target')
print(f'A = {obj_a["id"]}  bbox={obj_a["bbox"]}')
print(f'B = {obj_b["id"]}  bbox={obj_b["bbox"]}')

r = dispatch('hover_object', {'object_id': obj_b['id']}, ui_graph=g)
print('hover_object:', r)

A = obj_001  bbox=[704, 396, 84, 42]
B = obj_002  bbox=[708, 515, 84, 42]
  [L0] hover_object('obj_002') → moveTo (750,536)
hover_object: {'status': 'ok', 'tool': 'hover_object', 'object_id': 'obj_002', 'at': [750, 536], 'level': 0}


In [26]:
# --- 7. Connect B → A by dragging from B's edge anchor to A ---------
# connect_shapes computes the source anchor automatically: it picks the
# B-side that faces A (here, A is north of B so source_anchor='n').
r = dispatch(
    'connect_shapes',
    {'source_id': obj_b['id'], 'target_id': obj_a['id'], 'source_anchor': 'auto'},
    ui_graph=g,
)
print('connect_shapes:', r)

print('\n--- scene_graph after connect B → A ---')
show_scene()

  [L0] connect_shapes(obj_002→obj_001) drag (750,503) → (746,417)
connect_shapes: {'status': 'ok', 'tool': 'connect_shapes', 'source': 'obj_002', 'target': 'obj_001', 'source_anchor': 'n', 'target_anchor': 's', 'from': [750, 503], 'to': [746, 417], 'level': 0}

--- scene_graph after connect B → A ---
**Objects (2):**
  - `obj_001` Rectangle "Source"  bbox=[704,396,84x42]
  - `obj_002` Rectangle "Target"  bbox=[708,515,84x42]  *SELECTED*

**Edges (1):**
  - `edge_001` `obj_002`.n → `obj_001`.s

_(scene_graph op #8, last op: connect_shapes)_


In [27]:
# --- 8. Final state ---------------------------------------------------
# Persisted to state/scene_graph.json — the framework reads this back
# on next session, and the Executor includes it in the prompt under
# the 'SCENE GRAPH' heading.
print('Persisted state/scene_graph.json:')
with open(os.path.join(config.raw().get('paths', {}).get('state_dir', 'state'), 'scene_graph.json')) as f:
    print(json.dumps(json.load(f), indent=2))

Persisted state/scene_graph.json:
{
  "version": 1,
  "next_object_id": 3,
  "next_edge_id": 2,
  "objects": [
    {
      "id": "obj_001",
      "type": "Rectangle",
      "label": "Source",
      "bbox": [
        704,
        396,
        84,
        42
      ],
      "anchors": {
        "n": [
          746,
          396
        ],
        "s": [
          746,
          438
        ],
        "e": [
          788,
          417
        ],
        "w": [
          704,
          417
        ]
      },
      "selected": false,
      "created_op": 1,
      "last_updated_op": 4
    },
    {
      "id": "obj_002",
      "type": "Rectangle",
      "label": "Target",
      "bbox": [
        708,
        515,
        84,
        42
      ],
      "anchors": {
        "n": [
          750,
          515
        ],
        "s": [
          750,
          557
        ],
        "e": [
          792,
          536
        ],
        "w": [
          708,
          536
        ]
      },
   

## What just happened

Every operand call went through `dispatch()`, which routes to a small Python function in `core/tools/primitives.py`. None of these decisions — *which* handle to grab, *where* to drop, *which* scene-graph object to update — were made by an LLM. The framework owns them deterministically, using:

1. **`core/perception/handles.py`** — CV detection of the cyan resize dots, dark extend arrows, and rotation icon around the selected shape.
2. **`core/state/scene_graph.py`** — the typed in-memory + on-disk model: objects (id, type, label, bbox, anchors) and edges (source, target, anchor pair).
3. **`_scan_and_reconcile()`** in `primitives.py` — the inter-operation scanner. It runs after geometry-changing operands, re-detects the selection bbox, and updates the matching scene-graph object so the next operation reasons about an accurate canvas state.

When the Executor agent is involved, `core/agents/executor.py:build_prompt()` includes the scene-graph summary verbatim in the system prompt under the `## SCENE GRAPH` heading — the LLM sees object ids, types, labels, and bboxes, and decides *what* to do; the framework decides *how*.